In [ ]:
def extract_sample_embeddings(sample_index):
    """Extract the embeddings of a given sample from all layers.
    Returns an array of shape (num_layers, embedding_dim).
    """
    return layer_embeddings[:, sample_index, :]

def get_trajectory(sample_index, perplexity=5):

    sample_embeddings = extract_sample_embeddings(sample_index)
    scaler = StandardScaler()
    sample_embeddings_scaled = scaler.fit_transform(sample_embeddings)
    tsne = TSNE(n_components=2, perplexity=perplexity, random_state=42)
    traj_2d = tsne.fit_transform(sample_embeddings_scaled)
    return traj_2d

# Update marker icons and colors for trajectory visualization (same as grid)
icons_traj = {'Activation': 'o', 'Inhibition': 's', 'Phosphorylation': '^', 'Incorrect': 'x'}
colors_traj = {'Activation': 'blue', 'Inhibition': 'red', 'Phosphorylation': 'black', 'Incorrect': 'purple'}

# Specify sample indices for each relationship
activation_sample_index = 5
inhibition_sample_index = 37
phosphorylation_sample_index = 65

print(f"Selected Activation sample index: {activation_sample_index}")
print(f"Selected Inhibition sample index: {inhibition_sample_index}")
print(f"Selected Phosphorylation sample index: {phosphorylation_sample_index}")

# Extract trajectories for each specified sample
traj_activation_2d = get_trajectory(activation_sample_index)
traj_inhibition_2d = get_trajectory(inhibition_sample_index)
traj_phosphorylation_2d = get_trajectory(phosphorylation_sample_index)

# Create one figure with three subplots in one row for the 2D trajectories
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
sample_info = [
    ("Activation", traj_activation_2d, colors_traj['Activation'], icons_traj['Activation']),
    ("Inhibition", traj_inhibition_2d, colors_traj['Inhibition'], icons_traj['Inhibition']),
    ("Phosphorylation", traj_phosphorylation_2d, colors_traj['Phosphorylation'], icons_traj['Phosphorylation'])
]

for ax, (label, traj, color, marker) in zip(axes, sample_info):
    # Plot the trajectory line with markers
    ax.plot(traj[:, 0], traj[:, 1], '-' + marker, color=color, linewidth=2, markersize=8)

    # Annotate each point with its layer number (starting at 1)
    for i, (x, y) in enumerate(traj):
        ax.annotate(str(i + 1), (x, y), textcoords="offset points", xytext=(3, 3),
                    fontsize=8, color='black')

    ax.set_title(f"{label}", fontsize=14)
    ax.set_xlabel("t-SNE-1", fontsize=12)
    ax.set_ylabel("t-SNE-2", fontsize=12)
    ax.grid(True, linestyle='--', alpha=0.5)

    # Flip only the Activation subplot's y-axis
    if label == "Activation":
        ax.invert_yaxis()

plt.tight_layout()
plt.show()


layers_arr = np.arange(1, traj_activation_2d.shape[0] + 1)
df_activation = pd.DataFrame(traj_activation_2d, columns=["t-SNE-1", "t-SNE-2"])
df_activation["Layer"] = layers_arr
df_activation = df_activation[["Layer", "t-SNE-1", "t-SNE-2"]]

df_inhibition = pd.DataFrame(traj_inhibition_2d, columns=["t-SNE-1", "t-SNE-2"])
df_inhibition["Layer"] = layers_arr
df_inhibition = df_inhibition[["Layer", "t-SNE-1", "t-SNE-2"]]

df_phosphorylation = pd.DataFrame(traj_phosphorylation_2d, columns=["t-SNE-1", "t-SNE-2"])
df_phosphorylation["Layer"] = layers_arr
df_phosphorylation = df_phosphorylation[["Layer", "t-SNE-1", "t-SNE-2"]]

output_file = "trajectory_excel.xlsx"
with pd.ExcelWriter(output_file) as writer:
    df_activation.to_excel(writer, sheet_name="Activation", index=False)
    df_inhibition.to_excel(writer, sheet_name="Inhibition", index=False)
    df_phosphorylation.to_excel(writer, sheet_name="Phosphorylation", index=False)

print(f"Trajectory data saved to {output_file}")

